# Classifier / Extractor / PaDiM 再現性チェック

`networks` で定義されている各モデルについて、次の2点を確認する。

1. **同一の入力 + 同一のシード** を与えたとき、毎回同じ出力になること（再現性）
2. **同一の入力 + 異なるシード** を与えたとき、異なる出力になること（シードが実際に重みへ反映されていること）
   - ただし乱数を使わない `resnet_50` だけは例外で、シードによらず同じ出力になることを期待値とする

対象:
- classifiers: `networks.list_classifiers()` が返す全モデル（`esn`, `bi_esn`, `bi_esn2d`, `positionwise_fcl`, `fcl`, `conv2d`, `reservoir_conv2d`）
- extractors: `networks.list_extractors()` が返す全モデル（`birc2d_base_v1`, `birc2d_base_v2`, `resnet_50`）
- padim: `networks.padim.PaDiM` が持つチャネル間引き（PaDiM の次元削減）

入力サイズやユニット数は実行時間を抑えるため小さめの値を使う（学習・精度の検証が目的ではなく、あくまで再現性のスモークテスト）。

In [1]:
import os
import sys
import warnings

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

import torch

import networks

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", DEVICE)

torch: 2.13.0+cu132
device: cuda


## 利用可能なモデル一覧

In [2]:
print("classifiers:", networks.list_classifiers())
print("extractors: ", networks.list_extractors())

classifiers: ['bi_esn', 'bi_esn2d', 'conv2d', 'esn', 'reservoir_conv2d']
extractors:  ['birc2d_base_v1', 'birc2d_base_v2', 'resnet_50']


## 再現性チェックの方針

- `Classifier.forward()` は線形読み出し（`LinearReadout`）を通すが、読み出し重みは学習前は常にゼロ初期化されるため、`forward()` の出力はシードによらず常にゼロになる。
  そのため classifier は読み出し前の特徴量を返す `.features()` を比較対象にする。
- `networks.modules.Reservoir` / `BiReservoir` / `BiReservoir2D` は内部で `torch.Generator().manual_seed(seed)` を使って重みを初期化するため、`seed` 引数だけで再現する。
- 一方 `FeatureExtractorV2`（`birc2d_base_v2`）の出力ヘッド（`nn.Linear(..., bias=False)`）は `seed` 引数を使わず、PyTorch のデフォルト初期化（グローバル RNG）に依存している。
  そのため、モデルを構築する直前で `torch.manual_seed(seed)` を呼んでグローバル RNG も揃えておく（`set_seed()`）。これを怠ると `birc2d_base_v2` だけ同一シードでも出力が一致しない（詳細は末尾の補足セクションで実演する）。
- `resnet_50` は ImageNet 事前学習済みの重みを読み込むだけで乱数を使わないため、**シードによらず同じ出力になること**を期待値とする（`compare(..., seed_dependent=False)`）。
  seed で変わるチャネル間引きは `networks.padim.PaDiM` 側にあるので、extractor とは別のセルでチェックする。

In [3]:
def set_seed(seed: int) -> None:
    """グローバル RNG をこの seed に揃える。モデル内部の Generator は seed 引数で別途揃う。"""
    torch.manual_seed(seed)


def compare(name: str, out_same, out_same2, out_diff, seed_dependent: bool = True) -> dict:
    """同一シード同士が一致し、異なるシードとは不一致になっているかを判定する。

    seed_dependent=False のモデル（乱数を使わない ImageNet 事前学習済み backbone など）は、
    異なるシードでも出力が一致することを期待値とする。
    """
    same_seed_matches = torch.allclose(out_same, out_same2, atol=1e-6)
    diff_seed_differs = not torch.allclose(out_same, out_diff, atol=1e-6)

    result = {
        "model": name,
        "seed_dependent": seed_dependent,
        "same_seed_matches": same_seed_matches,
        "diff_seed_differs": diff_seed_differs,
        "same_seed_maxdiff": (out_same - out_same2).abs().max().item(),
        "diff_seed_maxdiff": (out_same - out_diff).abs().max().item(),
        "passed": same_seed_matches and (diff_seed_differs == seed_dependent),
    }

    status = "OK" if result["passed"] else "NG"
    print(
        f"[{status}] {name:28s} "
        f"same-seed maxdiff={result['same_seed_maxdiff']:.3e}  "
        f"diff-seed maxdiff={result['diff_seed_maxdiff']:.3e}"
    )
    return result


results = []

## Classifiers の再現性チェック

In [ ]:
CLASSIFIER_INPUT_SHAPE = (1, 8, 8)  # (C, H, W)。MNIST を想定したグレースケールの小さい画像
NUM_CLASSES = 10

# モデルごとに特徴量の次元を決める引数名が違うため、名前で引く
# (patch_sizes / connectivity / leaky / spectral_radius / kernel_size はデフォルト値を使用)
CLASSIFIER_KWARGS = {
    "esn": dict(units=16),
    "bi_esn": dict(units=16),
    "bi_esn2d": dict(units=16),
    "positionwise_fcl": dict(units=16),
    "fcl": dict(units=16),
    "conv2d": dict(filters=16),
    "reservoir_conv2d": dict(num_reservoirs=3, units=8),
}

x_cls = torch.randn(4, *CLASSIFIER_INPUT_SHAPE)  # 一意の入力。このセル実行中は固定して使い回す

for name in networks.list_classifiers():
    kwargs = CLASSIFIER_KWARGS[name]

    set_seed(0)
    model_a = networks.build_classifier(name, CLASSIFIER_INPUT_SHAPE, NUM_CLASSES, seed=0, **kwargs).to(DEVICE).eval()

    set_seed(0)
    model_b = networks.build_classifier(name, CLASSIFIER_INPUT_SHAPE, NUM_CLASSES, seed=0, **kwargs).to(DEVICE).eval()

    set_seed(0)
    model_c = networks.build_classifier(name, CLASSIFIER_INPUT_SHAPE, NUM_CLASSES, seed=1, **kwargs).to(DEVICE).eval()

    with torch.no_grad():
        feat_a = model_a.features(x_cls.to(DEVICE))
        feat_b = model_b.features(x_cls.to(DEVICE))
        feat_c = model_c.features(x_cls.to(DEVICE))

    results.append(compare(f"classifier:{name}", feat_a, feat_b, feat_c))

## Extractors の再現性チェック

extractor はモデルごとに構造が異なるため、それぞれ小さめの構成を用意する。

In [24]:
EXTRACTOR_CONFIGS = {
    "birc2d_base_v1": dict(
        input_shape=(3, 16, 16),
        output_shape=(12, 16, 16),
        x=torch.randn(1, 3, 16, 16),
        kwargs=dict(sample_sizes=(8, 4, 2), N=2),
        seed_dependent=True,
    ),
    "birc2d_base_v2": dict(
        input_shape=(3, 16, 16),
        output_shape=(12, 16, 16),
        x=torch.randn(1, 3, 16, 16),
        kwargs=dict(sample_sizes=(16, 8, 4), N_block=3, N_subblock=1, base_filters=4),
        seed_dependent=True,
    ),
    "resnet_50": dict(
        input_shape=(3, 64, 64),
        output_shape=None,
        x=torch.randn(1, 3, 64, 64),
        kwargs=dict(),
        # ImageNet 事前学習済みの backbone は乱数を使わないため、出力は seed に依存しない
        # （seed で決まる PaDiM のチャネル間引きは networks.padim.PaDiM が持つ。次節でチェックする）
        seed_dependent=False,
    ),
}


def build_extractor(name: str, seed: int):
    """グローバル RNG も揃えたうえで extractor を構築する。"""
    cfg = EXTRACTOR_CONFIGS[name]
    set_seed(seed)

    return (
        networks.build_extractor(name, cfg["input_shape"], cfg["output_shape"], seed=seed, **cfg["kwargs"])
        .to(DEVICE)
        .eval()
    )


for name in networks.list_extractors():
    cfg = EXTRACTOR_CONFIGS[name]
    x_ext = cfg["x"].to(DEVICE)

    model_a = build_extractor(name, seed=0)
    model_b = build_extractor(name, seed=0)
    model_c = build_extractor(name, seed=1)

    with torch.no_grad():
        out_a = model_a(x_ext)
        out_b = model_b(x_ext)
        out_c = model_c(x_ext)

    results.append(compare(f"extractor:{name}", out_a, out_b, out_c, seed_dependent=cfg["seed_dependent"]))

[OK] extractor:birc2d_base_v1     same-seed maxdiff=0.000e+00  diff-seed maxdiff=1.344e+00
[OK] extractor:birc2d_base_v2     same-seed maxdiff=0.000e+00  diff-seed maxdiff=9.464e-01
[OK] extractor:resnet_50          same-seed maxdiff=0.000e+00  diff-seed maxdiff=0.000e+00


## PaDiM（チャネル間引き）の再現性チェック

`networks.padim.PaDiM` は特徴抽出器が出したチャネルを `seed` で決まるインデックスでランダムに間引く（PaDiM の次元削減）。
ここでは backbone 自体が seed に依存しない `resnet_50` を使い、**間引きだけ**が seed で変わることを確認する。

In [25]:
PADIM_INPUT_SHAPE = (3, 64, 64)
PADIM_RANDOM_DIM_SIZE = 16

x_padim = torch.randn(2, *PADIM_INPUT_SHAPE).to(DEVICE)


def build_padim(seed: int):
    set_seed(seed)
    extractor = networks.build_extractor("resnet_50", PADIM_INPUT_SHAPE, None)

    return networks.PaDiM(extractor, random_dim_size=PADIM_RANDOM_DIM_SIZE, seed=seed).to(DEVICE).eval()


with torch.no_grad():
    emb_a = build_padim(seed=0).embed(x_padim)
    emb_b = build_padim(seed=0).embed(x_padim)
    emb_c = build_padim(seed=1).embed(x_padim)

results.append(compare("padim:sampling", emb_a, emb_b, emb_c))

[OK] padim:sampling               same-seed maxdiff=0.000e+00  diff-seed maxdiff=6.162e+00


## 結果まとめ

In [7]:
header = f"{'model':<28s} {'seed_dependent':>15s} {'same_seed_matches':>18s} {'diff_seed_differs':>18s} {'passed':>8s}"
print(header)
print("-" * len(header))
for r in results:
    print(
        f"{r['model']:<28s} {str(r['seed_dependent']):>15s} {str(r['same_seed_matches']):>18s} "
        f"{str(r['diff_seed_differs']):>18s} {str(r['passed']):>8s}"
    )

assert all(r["passed"] for r in results), "再現性チェックに失敗したモデルがあります。上の表を確認してください。"
print("\nすべてのモデルで再現性チェックに成功しました。")

model                         seed_dependent  same_seed_matches  diff_seed_differs   passed
-------------------------------------------------------------------------------------------
classifier:bi_esn                       True               True               True     True
classifier:bi_esn2d                     True               True               True     True
classifier:conv2d                       True               True               True     True
classifier:esn                          True               True               True     True
classifier:reservoir_conv2d             True               True               True     True
extractor:birc2d_base_v1                True               True               True     True
extractor:birc2d_base_v2                True               True               True     True
extractor:resnet_50                    False               True              False     True
padim:sampling                          True               True               Tr

## 補足: `torch.manual_seed()` を固定しない場合の挙動（`birc2d_base_v2` の注意点）

`networks.modules.Reservoir` / `BiReservoir` / `BiReservoir2D` は `seed` 引数だけで重みが決まる一方、`FeatureExtractorV2`（`birc2d_base_v2`）の出力ヘッド（`nn.Linear`, `bias=False`）は `seed` 引数を使わず PyTorch のデフォルト初期化（グローバル RNG）に依存している。

そのため `set_seed()`（`torch.manual_seed()`）を呼ばずに `seed=0` だけで2回モデルを構築すると、`birc2d_base_v2` は毎回異なる出力になる。以下のセルで再現する。

In [26]:
cfg = EXTRACTOR_CONFIGS["birc2d_base_v2"]
x_ext = cfg["x"].to(DEVICE)

# NOTE: あえて set_seed() を呼ばない（= torch のグローバル RNG を固定しない）
model_a = (
    networks.build_extractor("birc2d_base_v2", cfg["input_shape"], cfg["output_shape"], seed=0, **cfg["kwargs"])
    .to(DEVICE)
    .eval()
)
model_b = (
    networks.build_extractor("birc2d_base_v2", cfg["input_shape"], cfg["output_shape"], seed=0, **cfg["kwargs"])
    .to(DEVICE)
    .eval()
)

with torch.no_grad():
    out_a = model_a(x_ext)
    out_b = model_b(x_ext)

print("同じ seed=0 でも出力が一致する?     ", torch.allclose(out_a, out_b, atol=1e-6))
print(
    "出力ヘッド (nn.Linear) の重みが一致する?", torch.equal(model_a.heads["0"][0].weight, model_b.heads["0"][0].weight)
)

同じ seed=0 でも出力が一致する?      False
出力ヘッド (nn.Linear) の重みが一致する? False
